## Imports and Paths

In [1]:
# Import the libraries required for data preparation and analysis.
# Pandas will be used for loading, cleaning, transforming,
# and combining the datasets.

import pandas as pd
from pathlib import Path

# Define the project root directory.
# This allows us to use relative paths instead of hard-coded
# computer-specific paths.

PROJECT_ROOT = Path.cwd().parent

# Define the location of the raw datasets.
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"

# Display the path being used so we can verify it.
print(f"Raw data path: {RAW_DATA_PATH}")

Raw data path: D:\RetailIQ\data\raw


## Load the datasets we'll actually need

In [2]:
# Load the datasets required for the data preparation stage.
# The raw CSV files remain unchanged; we work with copies in memory.

orders = pd.read_csv(RAW_DATA_PATH / "olist_orders_dataset.csv")
order_items = pd.read_csv(RAW_DATA_PATH / "olist_order_items_dataset.csv")
products = pd.read_csv(RAW_DATA_PATH / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW_DATA_PATH / "olist_sellers_dataset.csv")
payments = pd.read_csv(RAW_DATA_PATH / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW_DATA_PATH / "olist_order_reviews_dataset.csv")
category_translation = pd.read_csv(
    RAW_DATA_PATH / "product_category_name_translation.csv"
)

# Confirm that all required datasets have been loaded.
print("All required datasets loaded successfully.")

All required datasets loaded successfully.


## Create our working copy

In [3]:
# Create independent copies of the raw datasets.
# This gives us working DataFrames that can be modified safely
# without affecting the original data loaded above.

orders_clean = orders.copy()
order_items_clean = order_items.copy()
products_clean = products.copy()
sellers_clean = sellers.copy()
payments_clean = payments.copy()
reviews_clean = reviews.copy()
category_translation_clean = category_translation.copy()

print("Working copies created successfully.")

Working copies created successfully.


## Standardize the date columns

In [4]:
# Define the date columns in the orders dataset.
# These columns currently contain date/time values stored as text.

order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# Convert each date column from text to pandas datetime.
# errors="coerce" converts invalid or missing values into NaT
# instead of causing the conversion to fail.

for column in order_date_columns:
    orders_clean[column] = pd.to_datetime(
        orders_clean[column],
        errors="coerce"
    )

# Display the resulting data types to verify the conversion.
print(orders_clean[order_date_columns].dtypes)

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [5]:
# Display a few records from the converted date columns.
# This lets us visually confirm that the values are being interpreted
# as actual dates and timestamps.

display(orders_clean[order_date_columns].head())

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


## Investigate missing dates

In [6]:
# Check missing values in the converted order date columns.
# We will compare missing delivery-related dates with order status
# before deciding how they should be handled.

missing_dates = (
    orders_clean[order_date_columns]
    .isna()
    .sum()
    .rename("Missing Values")
    .to_frame()
)

# Calculate the percentage of missing values in each date column.
missing_dates["Missing %"] = (
    missing_dates["Missing Values"]
    / len(orders_clean)
    * 100
).round(2)

# Display the missing-value summary.
display(missing_dates)

,Missing Values,Missing %
order_purchase_timestamp,0,0.00
order_approved_at,160,0.16
order_delivered_carrier_date,1783,1.79
order_delivered_customer_date,2965,2.98
order_estimated_delivery_date,0,0.00


In [7]:
# Compare missing delivery dates with the order status.
# This helps us determine whether missing dates are expected
# for orders that were not completed or delivered.

delivery_columns = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

for column in delivery_columns:
    print(f"\n{'=' * 70}")
    print(f"Missing values in: {column}")
    print(f"{'=' * 70}")
    
    # Select orders where the current date column is missing
    missing_date_orders = orders_clean[
        orders_clean[column].isna()
    ]
    
    # Count the order statuses among those records
    status_counts = (
        missing_date_orders["order_status"]
        .value_counts()
        .rename_axis("Order Status")
        .reset_index(name="Orders")
    )
    
    display(status_counts)


Missing values in: order_approved_at


,Order Status,Orders
0,canceled,141
1,delivered,14
2,created,5



Missing values in: order_delivered_carrier_date


,Order Status,Orders
0,unavailable,609
1,canceled,550
2,invoiced,314
3,processing,301
4,created,5
5,approved,2
6,delivered,2



Missing values in: order_delivered_customer_date


,Order Status,Orders
0,shipped,1107
1,canceled,619
2,unavailable,609
3,invoiced,314
4,processing,301
5,delivered,8
6,created,5
7,approved,2


In [8]:
# Check the missing values in the products dataset.
# This helps us understand which product attributes need attention
# before we use the data for analysis.

product_missing = (
    products_clean.isna()
    .sum()
    .rename("Missing Values")
    .to_frame()
)

# Calculate the percentage of missing values in each column.
product_missing["Missing %"] = (
    product_missing["Missing Values"]
    / len(products_clean)
    * 100
).round(2)

# Display only columns that contain missing values.
display(
    product_missing[
        product_missing["Missing Values"] > 0
    ]
)

,Missing Values,Missing %
product_category_name,610,1.85
product_name_lenght,610,1.85
product_description_lenght,610,1.85
product_photos_qty,610,1.85
product_weight_g,2,0.01
product_length_cm,2,0.01
product_height_cm,2,0.01
product_width_cm,2,0.01


In [9]:
# Check numeric product attributes for invalid values.
# Values such as zero or negative measurements may indicate
# data-quality issues that need investigation.

numeric_product_columns = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

# Count zero and negative values in each numeric column.
numeric_quality = []

for column in numeric_product_columns:
    numeric_quality.append({
        "Column": column,
        "Zero Values": (products_clean[column] == 0).sum(),
        "Negative Values": (products_clean[column] < 0).sum(),
        "Missing Values": products_clean[column].isna().sum()
    })

# Convert the results into a DataFrame for easy review.
numeric_quality = pd.DataFrame(numeric_quality)

# Display the numeric quality summary.
display(numeric_quality)

,Column,Zero Values,Negative Values,Missing Values
0,product_name_lenght,0,0,610
1,product_description_lenght,0,0,610
2,product_photos_qty,0,0,610
3,product_weight_g,4,0,2
4,product_length_cm,0,0,2
5,product_height_cm,0,0,2
6,product_width_cm,0,0,2


In [10]:
# Replace physically impossible zero product weights with missing values.
# We do not estimate the weight because the correct value is unknown.

products_clean.loc[
    products_clean["product_weight_g"] == 0,
    "product_weight_g"
] = pd.NA

# Confirm that no zero weights remain.
print(
    "Zero product weights remaining:",
    (products_clean["product_weight_g"] == 0).sum()
)

Zero product weights remaining: 0


In [11]:
# Add English product category names to the products dataset.
# The translation table provides English names for most of the
# Portuguese product categories.

products_clean = products_clean.merge(
    category_translation_clean,
    on="product_category_name",
    how="left"
)

# Rename the translated column to make its purpose clear.
products_clean = products_clean.rename(
    columns={
        "product_category_name_english": "product_category_english"
    }
)

# Display the category columns to verify the merge.
display(
    products_clean[
        [
            "product_category_name",
            "product_category_english"
        ]
    ].head(10)
)

,product_category_name,product_category_english
0,perfumaria,perfumery
1,artes,art
2,esporte_lazer,sports_leisure
3,bebes,baby
4,utilidades_domesticas,housewares
5,instrumentos_musicais,musical_instruments
6,cool_stuff,cool_stuff
7,moveis_decoracao,furniture_decor
8,eletrodomesticos,home_appliances
9,brinquedos,toys


## Clean the column names

In [12]:
# Standardize column names in the products dataset.
# The original dataset contains spelling errors in two column names.
# Correcting them makes the dataset easier to work with consistently.

products_clean = products_clean.rename(
    columns={
        "product_name_lenght": "product_name_length",
        "product_description_lenght": "product_description_length"
    }
)

# Display the updated column names to verify the changes.
print(products_clean.columns.tolist())

['product_id', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_english']


In [13]:
# Combine order items with the main order information.
# The result remains at the order-item level:
# one row represents one product item within an order.

order_item_data = order_items_clean.merge(
    orders_clean[
        [
            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]
    ],
    on="order_id",
    how="left"
)

# Add product information to each order item.
# This gives us the product category and product attributes
# associated with each item sold.

order_item_data = order_item_data.merge(
    products_clean[
        [
            "product_id",
            "product_category_name",
            "product_category_english",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ],
    on="product_id",
    how="left"
)

# Display the resulting structure.
print(f"Rows: {len(order_item_data):,}")
print(f"Columns: {len(order_item_data.columns)}")

display(order_item_data.head())

Rows: 112,650
Columns: 20


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,product_category_name,product_category_english,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,cool_stuff,cool_stuff,650.0,28.0,9.0,14.0
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,pet_shop,pet_shop,30000.0,50.0,30.0,40.0
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,moveis_decoracao,furniture_decor,3050.0,33.0,13.0,33.0
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,perfumaria,perfumery,200.0,16.0,10.0,15.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,ferramentas_jardim,garden_tools,3750.0,35.0,40.0,30.0


In [14]:
# Create the core financial metrics for RetailIQ.
# Sales represents the product price, while freight is kept separate.
# Order value represents the combined product price and freight.

order_item_data["sales"] = order_item_data["price"]

order_item_data["freight"] = order_item_data["freight_value"]

order_item_data["order_value"] = (
    order_item_data["sales"]
    + order_item_data["freight"]
)

# Display the newly created financial fields.
display(
    order_item_data[
        ["price", "freight_value", "sales", "freight", "order_value"]
    ].head()
)

,price,freight_value,sales,freight,order_value
0,58.90,13.29,58.90,13.29,72.19
1,239.90,19.93,239.90,19.93,259.83
2,199.00,17.87,199.00,17.87,216.87
3,12.99,12.79,12.99,12.79,25.78
4,199.90,18.14,199.90,18.14,218.04


In [15]:
# Validate the financial calculations.
# The totals should match the original order-item price and freight totals.

financial_check = pd.DataFrame({
    "Metric": [
        "Original Product Price",
        "RetailIQ Sales",
        "Original Freight",
        "RetailIQ Freight",
        "RetailIQ Order Value"
    ],
    "Total": [
        order_item_data["price"].sum(),
        order_item_data["sales"].sum(),
        order_item_data["freight_value"].sum(),
        order_item_data["freight"].sum(),
        order_item_data["order_value"].sum()
    ]
})

display(financial_check)

,Metric,Total
0,Original Product Price,13591643.70
1,RetailIQ Sales,13591643.70
2,Original Freight,2251909.54
3,RetailIQ Freight,2251909.54
4,RetailIQ Order Value,15843553.24


In [16]:
# Calculate delivery-performance metrics.
# These metrics will later support RetailIQ's delivery analysis
# and help identify delayed orders.

order_item_data["delivery_days"] = (
    order_item_data["order_delivered_customer_date"]
    - order_item_data["order_purchase_timestamp"]
).dt.total_seconds() / (60 * 60 * 24)

order_item_data["estimated_delivery_days"] = (
    order_item_data["order_estimated_delivery_date"]
    - order_item_data["order_purchase_timestamp"]
).dt.total_seconds() / (60 * 60 * 24)

order_item_data["delivery_delay_days"] = (
    order_item_data["delivery_days"]
    - order_item_data["estimated_delivery_days"]
)

# Display the calculated delivery metrics.
display(
    order_item_data[
        [
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "delivery_days",
            "estimated_delivery_days",
            "delivery_delay_days"
        ]
    ].head(10)
)

,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,estimated_delivery_days,delivery_delay_days
0,2017-09-13 08:59:02,2017-09-20 23:43:48,2017-09-29,7.614421,15.625671,-8.011250
1,2017-04-26 10:53:06,2017-05-12 16:04:24,2017-05-15,16.216181,18.546458,-2.330278
2,2018-01-14 14:33:31,2018-01-22 13:19:16,2018-02-05,7.948437,21.393391,-13.444954
3,2018-08-08 10:00:35,2018-08-14 13:32:39,2018-08-20,6.147269,11.582928,-5.435660
4,2017-02-04 13:57:51,2017-03-01 16:42:31,2017-03-17,25.114352,40.418160,-15.303808
5,2017-05-15 21:42:34,2017-05-22 13:44:35,2017-06-06,6.668067,21.095440,-14.427373
6,2017-12-10 11:53:48,2017-12-18 22:03:38,2018-01-04,8.423495,24.504306,-16.080810
7,2018-07-04 12:08:27,2018-07-09 14:04:07,2018-07-25,5.080324,20.494132,-15.413808
8,2018-03-19 18:40:33,2018-03-29 18:17:31,2018-03-29,9.984005,9.221840,0.762164
9,2018-07-02 13:59:39,2018-07-04 17:28:31,2018-07-23,2.145046,20.416910,-18.271863


In [17]:
# Classify delivery performance based directly on the delivery delay.
# Negative values mean the order arrived before the estimated date.
# Zero means the order arrived on the estimated date.
# Positive values mean the order arrived after the estimated date.
# Orders without an actual delivery date remain "Not Delivered".

order_item_data["delivery_status"] = "Not Delivered"

order_item_data.loc[
    order_item_data["delivery_delay_days"] < 0,
    "delivery_status"
] = "Early"

order_item_data.loc[
    order_item_data["delivery_delay_days"] == 0,
    "delivery_status"
] = "On Time"

order_item_data.loc[
    order_item_data["delivery_delay_days"] > 0,
    "delivery_status"
] = "Delayed"

# Display the delivery-status distribution.
display(
    order_item_data["delivery_status"]
    .value_counts()
    .rename_axis("Delivery Status")
    .reset_index(name="Order Items")
)

,Delivery Status,Order Items
0,Early,101481
1,Delayed,8715
2,Not Delivered,2454


In [18]:
# Create time-based features from the order purchase timestamp.
# These fields will later support monthly sales analysis,
# seasonal analysis, and sales forecasting.

order_item_data["order_date"] = (
    order_item_data["order_purchase_timestamp"].dt.date
)

order_item_data["order_year"] = (
    order_item_data["order_purchase_timestamp"].dt.year
)

order_item_data["order_month"] = (
    order_item_data["order_purchase_timestamp"].dt.month
)

order_item_data["order_month_name"] = (
    order_item_data["order_purchase_timestamp"].dt.month_name()
)

order_item_data["order_year_month"] = (
    order_item_data["order_purchase_timestamp"].dt.to_period("M")
    .astype(str)
)

# Display the new time features.
display(
    order_item_data[
        [
            "order_purchase_timestamp",
            "order_date",
            "order_year",
            "order_month",
            "order_month_name",
            "order_year_month"
        ]
    ].head()
)

,order_purchase_timestamp,order_date,order_year,order_month,order_month_name,order_year_month
0,2017-09-13 08:59:02,2017-09-13,2017,9,September,2017-09
1,2017-04-26 10:53:06,2017-04-26,2017,4,April,2017-04
2,2018-01-14 14:33:31,2018-01-14,2018,1,January,2018-01
3,2018-08-08 10:00:35,2018-08-08,2018,8,August,2018-08
4,2017-02-04 13:57:51,2017-02-04,2017,2,February,2017-02


In [19]:
# Add seller information to each order item.
# This gives us seller location data for later seller and regional analysis.

order_item_data = order_item_data.merge(
    sellers_clean[
        [
            "seller_id",
            "seller_zip_code_prefix",
            "seller_city",
            "seller_state"
        ]
    ],
    on="seller_id",
    how="left"
)

# Verify the seller information was added successfully.
print(f"Rows: {len(order_item_data):,}")
print(f"Columns: {len(order_item_data.columns)}")

display(
    order_item_data[
        [
            "seller_id",
            "seller_city",
            "seller_state"
        ]
    ].head()
)

Rows: 112,650
Columns: 35


,seller_id,seller_city,seller_state
0,48436dade18ac8b2bce089ec2a041202,volta redonda,SP
1,dd7ddc04e1b6c2c614352b383efe2d36,sao paulo,SP
2,5b51032eddd242adc84c38acab88f23d,borda da mata,MG
3,9d7a1d34a5052409006425275ba1c2b4,franca,SP
4,df560393f3a51e74553ab94004ba5c87,loanda,PR


In [20]:
# Aggregate payment information to one row per order.
# Some orders have multiple payment records, so aggregating first
# prevents duplicated order-item rows when we merge the payment data.

payment_summary = (
    payments_clean
    .groupby("order_id")
    .agg(
        total_payment_value=("payment_value", "sum"),
        payment_count=("payment_sequential", "count"),
        primary_payment_type=("payment_type", "first"),
        max_installments=("payment_installments", "max")
    )
    .reset_index()
)

# Display the payment summary.
print(f"Payment summary rows: {len(payment_summary):,}")

display(payment_summary.head())

Payment summary rows: 99,440


,order_id,total_payment_value,payment_count,primary_payment_type,max_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,credit_card,2
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,credit_card,3
2,000229ec398224ef6ca0657da4fc703e,216.87,1,credit_card,5
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,credit_card,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,credit_card,3


In [21]:
# Merge the aggregated payment information into the order-item dataset.
# Because payment_summary has one row per order, this will not duplicate
# the order-item records.

order_item_data = order_item_data.merge(
    payment_summary,
    on="order_id",
    how="left"
)

# Verify that the merge did not change the number of order-item rows.
print(f"Rows: {len(order_item_data):,}")
print(f"Columns: {len(order_item_data.columns)}")

display(
    order_item_data[
        [
            "order_id",
            "total_payment_value",
            "payment_count",
            "primary_payment_type",
            "max_installments"
        ]
    ].head()
)

Rows: 112,650
Columns: 39


,order_id,total_payment_value,payment_count,primary_payment_type,max_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1.0,credit_card,2.0
1,00018f77f2f0320c557190d7a144bdd3,259.83,1.0,credit_card,3.0
2,000229ec398224ef6ca0657da4fc703e,216.87,1.0,credit_card,5.0
3,00024acbcdf0a6daa1e931b038114c75,25.78,1.0,credit_card,2.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1.0,credit_card,3.0


In [22]:
# Aggregate review information to one row per order.
# This prevents duplicate order-item rows when review data is merged.

review_summary = (
    reviews_clean
    .groupby("order_id")
    .agg(
        review_score=("review_score", "mean"),
        review_count=("review_id", "count")
    )
    .reset_index()
)

# Display the review summary.
print(f"Review summary rows: {len(review_summary):,}")

display(review_summary.head())

Review summary rows: 98,673


,order_id,review_score,review_count
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1
1,00018f77f2f0320c557190d7a144bdd3,4.0,1
2,000229ec398224ef6ca0657da4fc703e,5.0,1
3,00024acbcdf0a6daa1e931b038114c75,4.0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,1


In [23]:
# Merge the aggregated review information into the order-item dataset.
# Because review_summary has one row per order, this will not duplicate
# the order-item records.

order_item_data = order_item_data.merge(
    review_summary,
    on="order_id",
    how="left"
)

# Verify that the merge did not change the number of order-item rows.
print(f"Rows: {len(order_item_data):,}")
print(f"Columns: {len(order_item_data.columns)}")

display(
    order_item_data[
        [
            "order_id",
            "review_score",
            "review_count"
        ]
    ].head()
)

Rows: 112,650
Columns: 41


,order_id,review_score,review_count
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1.0
1,00018f77f2f0320c557190d7a144bdd3,4.0,1.0
2,000229ec398224ef6ca0657da4fc703e,5.0,1.0
3,00024acbcdf0a6daa1e931b038114c75,4.0,1.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,1.0


In [24]:
# Final validation of the processed order-item dataset.
# Check the overall structure, missing values, and duplicate rows
# before saving the processed data.

print("Final dataset shape:")
print(f"Rows: {len(order_item_data):,}")
print(f"Columns: {len(order_item_data.columns)}")

print("\nDuplicate rows:")
print(order_item_data.duplicated().sum())

print("\nMissing values:")
final_missing = (
    order_item_data.isna()
    .sum()
    .to_frame("Missing Values")
)

final_missing["Missing %"] = (
    final_missing["Missing Values"]
    / len(order_item_data)
    * 100
).round(2)

display(
    final_missing[
        final_missing["Missing Values"] > 0
    ]
)

Final dataset shape:
Rows: 112,650
Columns: 41

Duplicate rows:
0

Missing values:


,Missing Values,Missing %
order_approved_at,15,0.01
order_delivered_carrier_date,1194,1.06
order_delivered_customer_date,2454,2.18
product_category_name,1603,1.42
product_category_english,1627,1.44
product_weight_g,26,0.02
product_length_cm,18,0.02
product_height_cm,18,0.02
product_width_cm,18,0.02
delivery_days,2454,2.18


In [26]:
# Save the final processed dataset.
# This file will be used for Stage 3 analysis and dashboard development.

processed_path = "../data/processed/retailiq_order_item_data.csv"

order_item_data.to_csv(
    processed_path,
    index=False
)

print(f"Processed dataset saved to: {processed_path}")

Processed dataset saved to: ../data/processed/retailiq_order_item_data.csv
